# 3_1. KNN ręcznie: od sąsiadów do klasyfikacji

**KNN** oznacza *k-nearest neighbors*, czyli `k` najbliższych sąsiadów.

W klasyfikacji KNN punkt testowy dostaje klasę przez głosowanie najbliższych sąsiadów:

$$
\hat y(x_*)=\operatorname{mode}\{y_i: x_i\in \mathcal{N}_k(x_*)\}.
$$

Główna intuicja:

- małe `k` → decyzja bardzo lokalna, podatna na szum,
- duże `k` → decyzja gładsza, ale może zgubić drobne lokalne struktury.


## W tym notebooku użyjemy

✓ Macierzy odległości D

✓ Macierzy kNN

## Do jakiej rodziny należy metoda?

PCA:
→ metody projekcyjne

t-SNE:
→ metody podobieństw

UMAP:
→ metody grafowe

KNN:
→ metody sąsiedztwa

DBSCAN:
→ metody gęstościowe

KMeans:
→ metody centroidowe

LDA:
→ metody dyskryminacyjne

SVM:
→ metody maksymalnego marginesu

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['axes.grid'] = True
pd.set_option('display.precision', 3)

def pairwise_distances_np(X):
    X = np.asarray(X, dtype=float)
    diff = X[:, None, :] - X[None, :, :]
    return np.sqrt((diff**2).sum(axis=2))

def as_df(M, labels):
    return pd.DataFrame(np.round(M, 3), index=labels, columns=labels)

def plot_points(X, labels, classes=None, title='', annotate=True, ax=None, test_points=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7,5))
    X = np.asarray(X)
    if classes is None:
        ax.scatter(X[:,0], X[:,1], s=100)
    else:
        for c in sorted(set(classes)):
            idx = np.array(classes) == c
            ax.scatter(X[idx,0], X[idx,1], s=100, label=str(c))
        ax.legend()
    if annotate:
        for lab, (x,y) in zip(labels, X):
            ax.text(x+0.04, y+0.04, lab, fontsize=12, weight='bold')
    if test_points:
        for lab, pt in test_points.items():
            ax.scatter([pt[0]], [pt[1]], s=160, marker='*')
            ax.text(pt[0]+0.06, pt[1]+0.06, lab, fontsize=12, weight='bold')
    ax.set_title(title)
    ax.set_aspect('equal', adjustable='box')
    return ax

def draw_graph(X, labels, A, title='', ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7,5))
    X = np.asarray(X)
    ax.scatter(X[:,0], X[:,1], s=120)
    for i, lab in enumerate(labels):
        ax.text(X[i,0]+0.04, X[i,1]+0.04, lab, fontsize=12, weight='bold')
    n = len(labels)
    for i in range(n):
        for j in range(n):
            if i != j and A[i,j] > 0:
                ax.plot([X[i,0], X[j,0]], [X[i,1], X[j,1]], alpha=0.5)
    ax.set_title(title)
    ax.set_aspect('equal', adjustable='box')
    return ax

def knn_adjacency(D, k):
    n = D.shape[0]
    A = np.zeros((n, n), dtype=int)
    for i in range(n):
        order = np.argsort(D[i])
        neigh = [j for j in order if j != i][:k]
        A[i, neigh] = 1
    return A

def symmetrize(A):
    return np.maximum(A, A.T)

def eps_adjacency(D, eps, include_self=False):
    A = (D <= eps).astype(int)
    if not include_self:
        np.fill_diagonal(A, 0)
    return A

from sklearn.neighbors import KNeighborsClassifier

X_train = np.array([
    [0.0, 0.0], [0.4, 0.2], [0.2, 0.7], [0.9, 0.3],
    [3.0, 2.8], [3.3, 3.2], [3.7, 2.9], [2.7, 3.4],
    [1.8, 1.5],  # punkt trochę mylący / graniczny
])
y_train = np.array([0,0,0,0, 1,1,1,1, 1])
labels_train = np.array([f'P{i}' for i in range(len(X_train))])
queries = {
    'X1': np.array([0.7, 0.5]),
    'X2': np.array([1.7, 1.5]),
    'X3': np.array([2.2, 2.0]),
}

plot_points(X_train, labels_train, classes=y_train, title='Dane treningowe i punkty testowe', test_points=queries)
plt.show()


## 1. Ręczne liczenie dla jednego punktu testowego

Weźmy punkt testowy:

$$
X_1=(0.7,0.5).
$$

Liczymy odległości od `X1` do wszystkich punktów treningowych:

$$
d(X_1,P_i)=\sqrt{(X_{1,1}-P_{i,1})^2+(X_{1,2}-P_{i,2})^2}.
$$


In [ ]:
def distances_to_query(X, q):
    return np.sqrt(((X - q)**2).sum(axis=1))

q = queries['X1']
dq = distances_to_query(X_train, q)
table = pd.DataFrame({
    'punkt': labels_train,
    'klasa': y_train,
    'odległość do X1': np.round(dq, 3)
}).sort_values('odległość do X1')
table


## 2. Głosowanie dla różnych `k`

Dla `k=1` patrzymy tylko na najbliższy punkt.

Dla `k=3` głosują trzej najbliżsi sąsiedzi.

Dla `k=5` głosuje pięciu najbliższych sąsiadów.


In [ ]:
def knn_vote_table(X, y, q, ks=(1,3,5,7)):
    d = distances_to_query(X, q)
    order = np.argsort(d)
    rows = []
    for k in ks:
        idx = order[:k]
        votes = pd.Series(y[idx]).value_counts().sort_index().to_dict()
        pred = max(votes, key=votes.get)
        rows.append({
            'k': k,
            'sąsiedzi': ', '.join(labels_train[idx]),
            'klasy sąsiadów': ', '.join(map(str, y[idx])),
            'głosy': votes,
            'predykcja': pred
        })
    return pd.DataFrame(rows)

knn_vote_table(X_train, y_train, queries['X1'])


## 3. Rysunek: których sąsiadów bierze KNN?

Poniżej ten sam punkt testowy `X1`, ale dla różnych wartości `k`.


In [ ]:
def plot_knn_neighbors(X, y, q, k, title):
    d = distances_to_query(X, q)
    idx = np.argsort(d)[:k]
    fig, ax = plt.subplots(figsize=(6,5))
    plot_points(X, labels_train, classes=y, ax=ax, title=title, test_points={'X': q})
    for j in idx:
        ax.plot([q[0], X[j,0]], [q[1], X[j,1]], linewidth=2, alpha=0.8)
    ax.scatter(X[idx,0], X[idx,1], s=260, facecolors='none', edgecolors='black', linewidths=2)
    plt.show()

for k in [1, 3, 5, 7]:
    plot_knn_neighbors(X_train, y_train, queries['X1'], k, f'KNN dla punktu X1, k={k}')


## 4. Granica decyzyjna dla różnych `k`

KNN nie uczy jawnego równania prostej. Granica decyzyjna wynika z lokalnego głosowania sąsiadów.

Dla małego `k` granica może być poszarpana. Dla większego `k` staje się gładsza.


In [ ]:
def plot_knn_boundary(k):
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_train, y_train)
    x_min, x_max = -0.5, 4.3
    y_min, y_max = -0.5, 4.2
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    fig, ax = plt.subplots(figsize=(6,5))
    ax.contourf(xx, yy, Z, alpha=0.25)
    plot_points(X_train, labels_train, classes=y_train, ax=ax, title=f'Granica decyzyjna KNN dla k={k}')
    plt.show()

for k in [1, 3, 5, 7]:
    plot_knn_boundary(k)


## 5. KNN jako graf sąsiedztwa

To jest most do UMAP i SCT-MAP.

Dla każdego punktu możemy zbudować graf:

$$
A^{(k)}_{ij}=\mathbf{1}\left[x_j\in \mathcal{N}_k(x_i)\right].
$$

Ta macierz jest zwykle kierunkowa. Możemy ją też symetryzować:

$$
A^{\mathrm{sym}}_{ij}=\max(A^{(k)}_{ij}, A^{(k)}_{ji}).
$$


In [ ]:
D_train = pairwise_distances_np(X_train)
for k in [1, 3, 5]:
    A_k = knn_adjacency(D_train, k)
    print(f'k={k}: kierunkowa macierz kNN')
    display(as_df(A_k, labels_train))
    fig, ax = plt.subplots(figsize=(6,5))
    draw_graph(X_train, labels_train, symmetrize(A_k), title=f'Symetryczny graf kNN, k={k}', ax=ax)
    plt.show()


## 6. Co warto zapamiętać?

KNN jest bardzo prosty, ale ważny koncepcyjnie:

```text
odległości → najbliżsi sąsiedzi → głosowanie albo graf
```

Ta sama idea wraca w UMAP, DBSCAN, analizie grafów sąsiedztwa i SCT-MAP.
